In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv("data/train.csv")
df.head()

In [ ]:
# removing unneccessary columns
df.drop(columns=['Name','Ticket','Cabin','PassengerId'],inplace = True)

In [ ]:
df.head()

In [ ]:
# creating new column family adding two columns SibSp+Parch
df['family'] = df['SibSp']+df['Parch']

In [ ]:
df.drop(columns=['SibSp','Parch'],inplace=True)
df.head()

In [ ]:
# checking null values
df.isnull().sum()

In [ ]:
# we can fill this null values after train test split using SimpleImputer class by sklearn

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
# hadling null values
si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy = 'most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked = si_embarked.transform(X_test[['Embarked']])

In [ ]:
# handling Categorical Column
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown = 'ignore')
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown = 'ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

X_test_sex = ohe_sex.fit_transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.fit_transform(X_test_embarked)

In [ ]:
# adding all things to train
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

X_train_final = np.concatenate((X_train_rem,X_train_age,X_train_sex,X_train_embarked),axis=1)
X_test_final = np.concatenate((X_test_rem,X_test_age,X_test_sex,X_test_embarked),axis=1)

In [ ]:
# making object of Decision Tree Classifier And fitting on data

clf = DecisionTreeClassifier()
clf.fit(X_train_final,y_train)

In [ ]:
#sklearn.ensemble.RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

clf_random = RandomForestClassifier()
clf_random.fit(X_train_final,y_train)

In [ ]:
y_predict_r = clf_random.predict(X_test_final)
y_predict_r

In [ ]:
y_predict = clf.predict(X_test_final)
y_predict

In [ ]:
acc = accuracy_score(y_test,y_predict)
print(f"Accuracy Score of Decsion Tree is {acc}")

In [ ]:
acc = accuracy_score(y_test,y_predict_r)
print(f"Accuracy Score of Random Forest is {acc}")

In [ ]:
import pickle

pickle.dump(ohe_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf_random,open('models/clf.pkl','wb'))

# Predicating the values 

In [ ]:
ohe_sex = pickle.load(open('models/ohe_sex.pkl','rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl','rb'))
clf = pickle.load(open('models/clf.pkl','rb'))

In [ ]:
user_input = (np.array([2,10.2,0,22,'male','S'],dtype=object)).reshape(1,6)

In [ ]:
ohe_sex_t = ohe_sex.transform(user_input[:,4].reshape(1,1))
ohe_sex_t

In [ ]:
ohe_embarked_t = ohe_embarked.transform(user_input[:,5].reshape(1,1))

In [ ]:
ohe_embarked_t

In [ ]:
user_input_rem = user_input[:,:4]
user_input_rem

In [ ]:
user_input_final = np.concatenate((user_input_rem,ohe_sex_t,ohe_embarked_t),axis=1)

In [ ]:
clf.predict(user_input_final)